# 4. Model Exporting (ONNX & IMX)

Export the trained model to ONNX or IMX deployment format.

**IMX Calibration Constraint**: Because exporting to `imx` requires Post-Training Quantization (PTQ), loading a huge validation dataset fills up memory and crashes the process. To solve this, we dynamically formulate a miniature configuration (`val_imx/` with 100 images and `data_imx.yaml`) strictly for exporting constraints.


In [ ]:
# Cross-Platform Environment Setup
import os
import sys

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print('Running in cloud environment. Installing dependencies...')
    !pip install -qU ultralytics wandb roboflow python-dotenv
    
    if IN_COLAB:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY') or ''
else:
    print('Running locally. Loading from .env...')
    try:
        from dotenv import load_dotenv
        # The .env file is usually in the root of the project (one level up from notebooks)
        load_dotenv('../.env')
    except ImportError:
        print('python-dotenv not installed. Please install it or set environment variables manually.')


In [ ]:
import os
import shutil
import random
import yaml
from ultralytics import YOLO

best_model_path = "../models/best_MSamir.pt"
model = YOLO(best_model_path)


In [ ]:
def create_imx_calibration_dataset(source_val_images_dir, source_val_labels_dir, num_images=100):
    """
    Copies a subset of the validation data into a lightweight `val_imx` folder to prevent OOM errors during INT8 PTQ.
    """
    dest_dir = '../val_imx'
    dest_images = os.path.join(dest_dir, 'images')
    dest_labels = os.path.join(dest_dir, 'labels')
    
    os.makedirs(dest_images, exist_ok=True)
    os.makedirs(dest_labels, exist_ok=True)
    
    # Get all images
    all_images = glob.glob(os.path.join(source_val_images_dir, '*.*'))
    # Limit to num_images
    selected_images = random.sample(all_images, min(num_images, len(all_images)))
    
    print(f'Copying {len(selected_images)} calibration files to {dest_dir}...')
    for img_path in selected_images:
        basename = os.path.basename(img_path)
        label_name = os.path.splitext(basename)[0] + '.txt'
        label_path = os.path.join(source_val_labels_dir, label_name)
        
        shutil.copy(img_path, os.path.join(dest_images, basename))
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(dest_labels, label_name))
            
    # Create a dummy data_imx.yaml that points to this minimal val_imx directory
    data_imx = {
        'path': os.path.abspath(dest_dir),
        'train': 'images',  # Ignored during export
        'val': 'images',    # Calibration source
        'names': {0: 'class1'} # REPLACE WITH REAL CLASSES
    }
    
    yaml_path = '../data_imx.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(data_imx, f)
        
    print(f'Successfully prepared calibration data layout at {yaml_path}.')
    return yaml_path

import glob
# Run helper:
# yaml_path = create_imx_calibration_dataset('../Datasets/target_dataset/images/val', '../Datasets/target_dataset/labels/val')


In [ ]:
# Export to IMX using the lightweight data_imx.yaml to avoid memory blowout
# model.export(format='imx', data='../data_imx.yaml')
model.export(format='imx')


In [ ]:
# Standard ONNX Export (Optional/Fallback)
# model.export(format='onnx', dynamic=False, opset=12)
